This notebook takes a folder which contains a set of csv files which are a list of candidate control sources for each pulsar, and removes duplicates while also removing sources that are too faint to be viable controls. After doing this, it knits together the separate dataframes for each pulsar in to one dataframe, which it then saves as a csv. In this case, that csv is 'new_cand_controls_truncated'. This notebook should be run after 'cut_controls'.

In [1]:
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units as u, constants as c
import pandas as pd
import numpy as np

In [2]:
psr_names = pd.read_csv('paper_df_VASTcoverage.csv')['JNAME']

In [2]:
container = pd.read_csv('cand_controls_bypsr/J1809-1943.csv')
container.drop(container.index, inplace=True)

In [3]:
container

,Unnamed: 0,psr_name,Unnamed: 1,name,ra,dec,skycoord,stokes,fields,primary_field,...,spectral_index_err,spectral_curvature_err,rms_image,has_siblings,fit_is_estimate,spectral_index_from_TT,flag_c4,comment,detection,distance


In [ ]:
for name in psr_names:
    print(name, end="\n")
    data = pd.read_csv('cand_controls_bypsr/' + name + '.csv')
    coords = SkyCoord(
        ra=data["ra_deg_cont"] * u.degree, dec=data["dec_deg_cont"] * u.degree
    )
    
    #gets rid of duplicate sources by looking at angular separation
    checked_inds = []
    for i in np.arange(coords.size):
        #print(str(i), end = ",")
        if i in checked_inds:
            continue
        for j in np.arange(coords.size):
            if j in checked_inds:
                continue
            if ((coords[i].separation(coords[j]).arcsec < 10) & (i != j)):
                checked_inds.append(j)
    inds_to_delete = checked_inds
    data.drop(inds_to_delete, inplace=True)

    # Select bright sources (SNR >=8)
    snr = data["flux_peak"] / data["rms_image"]

    snr_mask = snr >= 8
    
    #distance mask
    #dist_mask = data['distance'] < 1800
    
    # Filter the data
    cut_data = data[
        (snr_mask) 
    #    & (dist_mask)
    ]
    
    cut_data.to_csv('truncated_ctrls_bypsr/' + name + '.csv')
    container = pd.concat([container, cut_data])

J1654-3710
J1709-3626


In [16]:
for name in psr_names:
    if (container[(container['psr_name']==name)
                    #&(container['distance']<2400)
                    &(container['flux_peak']>7)]
          .shape[0]
         <30):
        print(name)

J1803-2137
J1708-3426


In [7]:
for name in psr_names:
    data = pd.read_csv('truncated_ctrls_bypsr/' + name + '.csv')
    data.drop(columns=data.columns[0], inplace=True)
    container = pd.concat([container, data])

In [13]:
container = container[container['flux_peak']>7]

In [17]:
container.to_csv('new_cand_controls_truncated.csv')